# 03 — Analysis suite

Run the analysis modules on a trained checkpoint: layer-wise p(C=1) decoding, the implicit-integration suite, the congruent/opposite + Bayes-factor mechanism, indices, geometry, and RF shifts.


In [2]:
from pathlib import Path
import numpy as np

from causal_msi.config import load_config
from causal_msi.io import load_checkpoint, load_dataset
from causal_msi.models import build_model

cfg = load_config(Path('..') / 'configs' / 'default.yaml')
payload = load_checkpoint('../checkpoints/model_seed0.pth')
model = build_model(int(payload['input_dim']), cfg.model)
model.load_state_dict(payload['state_dict']); model.eval()
dataset = load_dataset('../data/dataset.npz')

In [3]:
# Layer-wise decodability of analytical p(C=1) (decoder fitting is implemented).
from causal_msi.analysis.decoding import decode_pc_by_layer

res = decode_pc_by_layer(model, dataset.X, dataset.targets.p_common, cfg.analysis.decoder)
print('SIL R^2:', res.sil.r2, ' MSL R^2:', res.msl.r2)

SIL R^2: 0.08122028271526671  MSL R^2: 0.14580822556201312


In [6]:
# The sharper integration analysis (requires TODO(science) estimators).
from causal_msi.analysis import integration as integ
from causal_msi.analysis.decoding import extract_activations

analytical = integ.analytical_model_averaged_estimate(dataset.targets)
msl = extract_activations(model, dataset.X)['msl']
decoded = integ.population_decoded_estimate(msl, analytical, cfg.analysis.decoder)
print('population-decoded estimate R^2:', decoded.r2)

NotImplementedError: TODO(science): analytical model-averaged estimate